# 02 — Multi-Layer Perceptron (MLP) no MNIST

Neste notebook, avançamos do SLP para uma arquitetura com **camadas ocultas**.

Fluxo:
1. Carregar e pré-processar o MNIST
2. Construir um MLP com 2 camadas ocultas
3. Treinar com Adam + Cross Entropy
4. Avaliar e visualizar resultados


In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)
print('Seed fixa:', SEED)

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

X_train = X_train.reshape(-1, 28 * 28)
X_test = X_test.reshape(-1, 28 * 28)

y_train_ohe = tf.keras.utils.to_categorical(y_train, 10)
y_test_ohe = tf.keras.utils.to_categorical(y_test, 10)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(784,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train_ohe,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test_ohe, verbose=0)
print(f'Loss no teste     : {test_loss:.4f}')
print(f'Acurácia no teste : {test_acc:.2%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Treino')
axes[0].plot(history.history['val_loss'], label='Validação')
axes[0].set_title('MLP — Loss')
axes[0].set_xlabel('Época')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Treino')
axes[1].plot(history.history['val_accuracy'], label='Validação')
axes[1].set_title('MLP — Acurácia')
axes[1].set_xlabel('Época')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Blues')
plt.title('Matriz de confusão — MLP (MNIST)')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.colorbar()
plt.show()

print(classification_report(y_test, y_pred, digits=4))

## Conclusão

Com camadas ocultas e ativações não-lineares, o MLP consegue modelar relações mais complexas do que o SLP. No próximo passo da trilha, a CNN explora a estrutura espacial das imagens de forma ainda mais eficiente.
